In [30]:
import pandas as pd
import nltk
import time
from parallel_pandas import ParallelPandas
# Source - https://stackoverflow.com/a/79071153
# Posted by Carolina Goes, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-19, License - CC BY-SA 4.0

nltk.download('punkt_tab')
nltk.download('stopwords')

#from nltk.corpus import stopwords
#from nltk.tokenize import word_tokenize

from fakenews_functions import clean_data, tokenize_and_remove_stopwords, stemming_words
# import my_regex as mre

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hsfac\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hsfac\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
ParallelPandas.initialize(n_cpu=16,split_factor=16)

In [32]:
def fuf(data):
    total_counts=pd.Series()
    data: pd.Series =data
    data=data.p_apply(lambda s: s.split(" "))
    flattened=data.explode()
    counts=flattened.value_counts(sort=False)
    total_counts=total_counts.add(counts,fill_value=0)
    return total_counts
    

In [33]:
path_of_sample="E:\KU_MachineLearning_Study\Fake_news_project\Group-Fake-News-Project\\news_sample.csv"
sample_data = pd.read_csv(path_of_sample)
sample_content = sample_data["content"]
sample_content.dropna(inplace=True)

<>:1: SyntaxWarning: invalid escape sequence '\K'
<>:1: SyntaxWarning: invalid escape sequence '\K'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_21508\342605070.py:1: SyntaxWarning: invalid escape sequence '\K'
  path_of_sample="E:\KU_MachineLearning_Study\Fake_news_project\Group-Fake-News-Project\\news_sample.csv"


In [34]:
sample_content_cleaned = sample_content.apply(clean_data)

In [35]:
sample_content_tokenized = sample_content_cleaned.apply(tokenize_and_remove_stopwords)


In [36]:
sample_content_stemmed = sample_content_tokenized.apply(stemming_words)


In [50]:
print(len(fuf(sample_content_cleaned)))
print(len(fuf(pd.Series(sample_content_tokenized.explode()))))
print(len(fuf(sample_content_stemmed)))


<LAMBDA> DONE:   0%|          | 0/250 [00:00<?, ?it/s]

<LAMBDA> DONE:   0%|          | 0/250 [00:00<?, ?it/s]

content
!        902
#         20
$        352
%        332
&        412
        ... 
…they      2
′s        10
€          4
→          4
≥          2
Length: 11269, dtype: object


<LAMBDA> DONE:   0%|          | 0/250 [00:00<?, ?it/s]

25509


<LAMBDA> DONE:   0%|          | 0/127563 [00:00<?, ?it/s]

16555


<LAMBDA> DONE:   0%|          | 0/250 [00:00<?, ?it/s]

11269


In [77]:

file_chunks = pd.read_csv("E:\KU_MachineLearning_Study\Fake_news_project\995,000_rows (1).csv", usecols=["content"], chunksize=100000)
count_clean=pd.Series()
count_stop=pd.Series()
count_stem=pd.Series()
for chunk_number, chunk in enumerate(file_chunks):
    data: pd.Series = chunk["content"]
    data.dropna(inplace=True)
    
    
    data = data.p_apply(clean_data)
    temp_data: pd.Series= data
    temp_data = temp_data.p_apply(lambda s: s.split(" "))
    flattened = temp_data.explode()
    counts = flattened.value_counts(sort=False)
    count_clean = count_clean.add(counts, fill_value=0)
    
    
    # data.to_csv(f"./../data/data_cleaned/cleaned_chunk{chunk_number:02}.csv")
    
    
    data = data.p_apply(tokenize_and_remove_stopwords)
    
    temp_data: pd.Series= pd.Series(data.explode())
    temp_data = temp_data.p_apply(lambda s: s.split(" "))
    flattened = temp_data.explode()
    counts = flattened.value_counts(sort=False)
    count_stop = count_stop.add(counts, fill_value=0)
    
    
    # data.to_csv(f"./../data/data_tokenized_no_stopwords/tokenized_chunk{chunk_number:02}.csv")
    data = data.p_apply(stemming_words)
    temp_data: pd.Series= data
    temp_data = temp_data.p_apply(lambda s: s.split(" "))
    flattened = temp_data.explode()
    counts = flattened.value_counts(sort=False)
    count_stem = count_stem.add(counts, fill_value=0)
   
    
    print(count_clean)
    print(count_stop)
    print(count_stem)

    #data.to_csv(f"./../data/data_stemmed/stemmed_chunk{chunk_number:02}.csv")

<>:1: SyntaxWarning: invalid escape sequence '\K'
<>:1: SyntaxWarning: invalid escape sequence '\K'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_21508\289988514.py:1: SyntaxWarning: invalid escape sequence '\K'
  file_chunks = pd.read_csv("E:\KU_MachineLearning_Study\Fake_news_project\995,000_rows (1).csv", usecols=["content"], chunksize=100000)


CLEAN_DATA DONE:   0%|          | 0/99999 [00:00<?, ?it/s]

<LAMBDA> DONE:   0%|          | 0/99999 [00:00<?, ?it/s]

TOKENIZE_AND_REMOVE_STOPWORDS DONE:   0%|          | 0/99999 [00:00<?, ?it/s]

<LAMBDA> DONE:   0%|          | 0/37631644 [00:00<?, ?it/s]

...

TOKENIZE_AND_REMOVE_STOPWORDS DONE:   0%|          | 0/94999 [00:00<?, ?it/s]

<LAMBDA> DONE:   0%|          | 0/33520337 [00:00<?, ?it/s]

STEMMING_WORDS DONE:   0%|          | 0/94999 [00:00<?, ?it/s]

<LAMBDA> DONE:   0%|          | 0/94999 [00:00<?, ?it/s]

content
        56636.0
!        4135.0
!!        511.0
!!!       443.0
!!!!       95.0
         ...   
🦃           6.0
🦃🦃🦃         1.0
🦄,          1.0
🦅           2.0
🧐           1.0
Length: 4569899, dtype: object
content
!      503748.0
#      121824.0
$      847524.0
%      280417.0
&      171473.0
         ...   
🦃           6.0
🦃🦃🦃         1.0
🦄           1.0
🦅           2.0
🧐           1.0
Length: 2089905, dtype: object
content
!      503748.0
#      121824.0
$      847524.0
%      280417.0
&      171473.0
         ...   
🦃           6.0
🦃🦃🦃         1.0
🦄           1.0
🦅           2.0
🧐           1.0
Length: 1875210, dtype: object


In [79]:
print(len(count_clean))
print(len(count_stop))
print(len(count_stem))



4569899
2089905
1875210
